In [0]:
CATALOG = "cat_medallion"
SCHEMA  = "source_1"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.poc3_customer_source"
SCD1_TABLE   = f"{CATALOG}.{SCHEMA}.poc3_dim_customer_scd1"
SCD2_TABLE   = f"{CATALOG}.{SCHEMA}.poc3_dim_customer_scd2"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cat_medallion.source_1.poc3_dim_customer_scd1 (
    customer_id STRING NOT NULL,
    name        STRING,
    email       STRING,
    city        STRING
)
USING DELTA;

# Data Frame

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

batch1_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("name",        StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("ingest_ts",   StringType(), False),
    StructField("batch_seq",   IntegerType(), False),
])

batch1_data = [
    ("C001", "Raj",   "raj@old.com",     "Mumbai",    "2024-01-01 10:00:00", 1),
    ("C001", "Raj",   "raj@old.com",     "Mumbai",    "2024-01-01 10:00:05", 1),
    ("C002", "Priya", "priya@old.com",   "Delhi",     "2024-01-01 10:00:00", 1),
    ("C003", "Amit",  "amit@old.com",    "Bangalore", "2024-01-01 10:00:00", 1),
]

batch1_df = spark.createDataFrame(batch1_data, schema=batch1_schema) \
    .withColumn("ingest_ts", to_timestamp(col("ingest_ts"), "yyyy-MM-dd HH:mm:ss"))


display(batch1_df.orderBy("customer_id", "ingest_ts"))

# Remove duplicate

In [0]:
w=Window.partitionBy("customer_id").orderBy(col("ingest_ts").desc(),col("batch_seq").desc())
batch1_df = batch1_df.withColumn("rn",row_number().over(w)).filter(col("rn")==1).drop("rn")

# write data to source staging table

In [0]:
batch1_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SOURCE_TABLE)
display(spark.table(SOURCE_TABLE).orderBy("customer_id"))

# Initial load into SCD1

In [0]:
df_ini_scd1=spark.read.table(SOURCE_TABLE).select("customer_id","name","email","city")
df_ini_scd1.write.format("delta").mode("overwrite").saveAsTable(SCD1_TABLE)
display(spark.table(SCD1_TABLE).orderBy("customer_id"))

# changed data or batch 2 data

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

batch2_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("name",        StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("ingest_ts",   StringType(), False),
    StructField("batch_seq",   IntegerType(), False),
])

batch2_data = [
    ("C001", "Raj",   "raj@new.com",   "Pune",    "2024-02-01 10:00:00", 2),
    ("C002", "Priya", "priya@old.com", "Delhi",   "2024-02-01 10:00:00", 2),
    ("C004", "Neha",  "neha@new.com",  "Chennai", "2024-02-01 10:00:00", 2)
]

batch2_df = spark.createDataFrame(batch2_data, schema=batch1_schema) \
    .withColumn("ingest_ts", to_timestamp(col("ingest_ts"), "yyyy-MM-dd HH:mm:ss"))


display(batch2_df.orderBy("customer_id", "ingest_ts"))

# write batch 2 data to stage table

In [0]:
batch2_df.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable(SOURCE_TABLE)
display(spark.table(SOURCE_TABLE).orderBy("customer_id"))

# merge for scd1

In [0]:
%sql
MERGE INTO cat_medallion.source_1.poc3_dim_customer_scd1 AS t
USING (
    SELECT customer_id, name, email, city
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY customer_id
                   ORDER BY ingest_ts DESC
               ) AS rn
        FROM cat_medallion.source_1.poc3_customer_source
        WHERE batch_seq = 2
    )
    WHERE rn = 1
) AS s
ON t.customer_id = s.customer_id

WHEN MATCHED AND (
       NOT (t.name  <=> s.name)
    OR NOT (t.email <=> s.email)
    OR NOT (t.city  <=> s.city)
)
THEN UPDATE SET
    t.name  = s.name,
    t.email = s.email,
    t.city  = s.city

WHEN NOT MATCHED THEN
INSERT (customer_id, name, email, city)
VALUES (s.customer_id, s.name, s.email, s.city);